In [1]:
from pyspark.sql import functions as F

from multitudcsd.config import get_spark_session
from multitudcsd.storage import read_delta

sesion = get_spark_session("comprobacion-route-types")

In [2]:
bronze_routes = read_delta(sesion, "bronze", "bronze_gtfs_static_routes")
print(f"[comprobaciones] {bronze_routes.count()} lineas en bronze_gtfs_static_routes")

[comprobaciones] 145 lineas en bronze_gtfs_static_routes


In [3]:
inventario = (
    bronze_routes
    .groupBy("route_type")
    .agg(
        F.count("*").alias("num_lineas"),
        F.slice(F.sort_array(F.collect_set("route_short_name")), 1, 5).alias("ejemplos"),
    )
    .orderBy(F.col("num_lineas").desc())
)

inventario.show(50, truncate=False)

+----------+----------+----------------------------+
|route_type|num_lineas|ejemplos                    |
+----------+----------+----------------------------+
|700       |76        |[100, 101, 106, 109, 110]   |
|109       |38        |[S1, S15, S2, S25, S26]     |
|900       |11        |[12, 18, M1, M10, M2]       |
|100       |10        |[FEX, RB10, RB63, RE1, RE20]|
|400       |9         |[U1, U2, U3, U4, U5]        |
|106       |1         |[RB63]                      |
+----------+----------+----------------------------+



In [4]:
import json, pathlib
from multitudcsd.ingestion.http_request import download_json

payload = download_json("https://api.viz.berlin.de/tic3/baustellen_sperrungen_tic.json")
payload["features"] = payload["features"][:3]   # recortado a 3 incidencias
pathlib.Path("tests/fixtures/viz_disruptions_sample.json").write_text(
    json.dumps(payload, ensure_ascii=False), encoding="utf-8"
)

[http] OK https://api.viz.berlin.de/tic3/baustellen_sperrungen_tic.json (691053 bytes)


FileNotFoundError: [Errno 2] No such file or directory: 'tests\\fixtures\\viz_disruptions_sample.json'

In [5]:
import os

# El notebook se ejecuta desde notebooks/, asi que hay que asegurar el entorno local.
os.environ["ENV"] = "local"

from multitudcsd.config import get_lakehouse_root, get_spark_session
from multitudcsd.storage import get_table_path, read_delta

print(get_lakehouse_root())
print(get_table_path("bronze", "bronze_csd_mentions"))

D:/05_MasterUCM/TFM/multitudcsd/data/lakehouse
D:/05_MasterUCM/TFM/multitudcsd/data/lakehouse/bronze/bronze_csd_mentions


In [5]:
bronze_mentions = read_delta(sesion, "bronze", "bronze_csd_mentions")

print(f"filas totales: {bronze_mentions.count()}")
bronze_mentions.printSchema()

filas totales: 2000
root
 |-- mention_id: string (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- platform: string (nullable = true)
 |-- language: string (nullable = true)
 |-- sentiment: double (nullable = true)
 |-- has_media: boolean (nullable = true)
 |-- user_hash: string (nullable = true)
 |-- source: string (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- ingest_date: date (nullable = true)



In [6]:
bronze_mentions.orderBy("mention_id").show(10, truncate=False)

+----------+-------------------+---------+---------+--------+--------+---------+---------+----------------+---------+-----------------------+-----------+
|mention_id|event_ts           |lat      |lon      |platform|language|sentiment|has_media|user_hash       |source   |ingest_ts              |ingest_date|
+----------+-------------------+---------+---------+--------+--------+---------+---------+----------------+---------+-----------------------+-----------+
|men_000001|2026-09-05T13:40:45|52.508781|13.403951|mastodon|pl      |0.334    |true     |c27999ae441220a4|synthetic|2026-09-06 21:05:07.695|2026-09-06 |
|men_000002|2026-09-05T15:09:11|52.512071|13.400993|mastodon|de      |0.425    |false    |ace6292a2ce688fe|synthetic|2026-09-06 21:05:07.695|2026-09-06 |
|men_000003|2026-09-05T16:10:18|52.518426|13.380253|x       |pl      |0.579    |true     |597c82ced4ed3adf|synthetic|2026-09-06 21:05:07.695|2026-09-06 |
|men_000004|2026-09-05T14:54:03|52.518696|13.372924|mastodon|en      |0.172 

In [7]:
from pyspark.sql import functions as F

distintas = bronze_mentions.select("mention_id").distinct().count()
print(f"mention_id distintos: {distintas} de {bronze_mentions.count()} filas")

# Si no cuadran, es que el stream reproceso ficheros (checkpoint borrado o relanzado).
(
    bronze_mentions.groupBy("mention_id")
    .count()
    .filter(F.col("count") > 1)
    .show(5)
)

mention_id distintos: 2000 de 2000 filas
+----------+-----+
|mention_id|count|
+----------+-----+
+----------+-----+

